# Qwen3.8-27B 94K rollout topology decision

This notebook reproduces the content-free benchmark tables and the TP4/TP8 decision from the committed safe JSON evidence.

In [1]:
from pathlib import Path
import json
import pandas as pd

evidence_path = Path('qwen38_27b_topology_decision_20260817.safe.json')
if not evidence_path.is_file():
    evidence_path = Path('docs') / evidence_path
evidence = json.loads(evidence_path.read_text(encoding='utf-8'))
evidence['contract'], evidence['decision']['selected_topology']

('llin-qwen38-topology-decision-safe-evidence-v1', 'TP4xDP4')

In [2]:
runs = pd.DataFrame(evidence['runs'])
runs['timeout_rate'] = runs['trajectory_timeout_rows'] / runs['completed_rows']
display_columns = [
    'round', 'host', 'topology', 'completed_rows', 'wall_seconds',
    'response_tokens_per_hour', 'trajectories_per_hour',
    'trajectory_timeout_rows', 'timeout_rate', 'hbm_usage_max',
    'runtime_error_rows',
]
runs[display_columns].sort_values(['round', 'host']).reset_index(drop=True)

,round,host,topology,completed_rows,wall_seconds,response_tokens_per_hour,trajectories_per_hour,trajectory_timeout_rows,timeout_rate,hbm_usage_max,runtime_error_rows
0,1,m00,TP4xDP3,64,2422,1.032180e+06,95.127993,7,0.109375,83,0
1,1,m05,TP8xDP2,64,2047,1.194720e+06,112.554958,9,0.140625,85,0
2,1,m06,TP4xDP4,64,2027,1.414614e+06,113.665516,4,0.062500,83,0
3,2,m05,TP4xDP4,64,2029,1.294758e+06,113.553475,3,0.046875,83,0
4,2,m06,TP8xDP2,64,2030,1.406942e+06,113.497537,7,0.109375,84,0


In [3]:
cross = runs[runs['host'].isin(['m05', 'm06'])].copy()
topology_summary = (
    cross.groupby('topology', as_index=False)
    .agg(
        host_runs=('host', 'count'),
        mean_response_tokens_per_hour=('response_tokens_per_hour', 'mean'),
        mean_trajectories_per_hour=('trajectories_per_hour', 'mean'),
        total_timeouts=('trajectory_timeout_rows', 'sum'),
        completed_rows=('completed_rows', 'sum'),
        peak_hbm_usage=('hbm_usage_max', 'max'),
        runtime_error_rows=('runtime_error_rows', 'sum'),
    )
)
topology_summary['timeout_rate'] = topology_summary['total_timeouts'] / topology_summary['completed_rows']
topology_summary

,topology,host_runs,mean_response_tokens_per_hour,mean_trajectories_per_hour,total_timeouts,completed_rows,peak_hbm_usage,runtime_error_rows,timeout_rate
0,TP4xDP4,2,1.354686e+06,113.609495,7,128,83,0,0.054688
1,TP8xDP2,2,1.300831e+06,113.026248,16,128,85,0,0.125000


In [4]:
capacity = pd.DataFrame(evidence['capacity']).T.reset_index(names='topology')
capacity['headroom_at_configured_maxseq'] = (
    1 - capacity['configured_max_num_seqs'] / capacity['full_context_concurrency_per_replica']
)
capacity

,topology,npu_count,weight_gib_per_card,full_context_concurrency_per_replica,configured_max_num_seqs,headroom_at_configured_maxseq
0,TP8xDP2,16.0,6.5904,26.40,32.0,-0.212121
1,TP4xDP4,16.0,13.0092,20.93,16.0,0.235547
2,TP4xDP3,12.0,13.0092,20.93,16.0,0.235547


In [5]:
assert evidence['decision']['selected_topology'] == 'TP4xDP4'
assert (runs['exit_code'] == 0).all()
assert (runs['runtime_error_rows'] == 0).all()
assert (runs['completed_rows'] == 64).all()
assert evidence['decision']['tp4_maxseq20_headroom_fraction'] < 0.05
evidence['formal_launch']

{'stage': 'running_v15_wave2',
 'm05': {'topology': 'TP4xDP4',
  'npu_count': 16,
  'max_num_seqs': 16,
  'physical_sequence_capacity': 64,
  'logical_window_trajectories': 80,
  'tasks_per_version': 182,
  'total_tasks': 546},
 'm06': {'topology': 'TP4xDP4',
  'npu_count': 16,
  'max_num_seqs': 16,
  'physical_sequence_capacity': 64,
  'logical_window_trajectories': 80,
  'tasks_per_version': 182,
  'total_tasks': 546},
 'm00': {'topology': 'TP4xDP3',
  'npu_count': 12,
  'max_num_seqs': 16,
  'physical_sequence_capacity': 48,
  'logical_window_trajectories': 60,
  'tasks_per_version': 136,
  'total_tasks': 408},
 'version_order': ['v15', 'v20', 'v21'],
 'sampling_protocol': 'strict adaptive 2+2+2',
 'source_tasks_per_version': 500,
 'total_source_tasks': 1500,
 'training_allowed': False}